<a href="https://colab.research.google.com/github/rwcitek/TechEx-LangGraph-workshop/blob/main/notebooks/solutions/01_design_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 1 — System Design for Multi-Agent Workflows

> **Format:** Paper-and-keyboard. No LLM calls — we're designing, not implementing.
>
> **Time:** 15 minutes.

By the end of this notebook you'll have:

1. A typed `TriageState` schema that defines what flows between agents
2. A graph topology — which nodes exist, which edges connect them
3. A short "anti-pattern check" exercise to catch design mistakes before they become bugs

Implementation comes in Module 2.

> *This is the solution notebook. Open the starter first; come here only after you've tried it yourself.*

## Part A  —  The state schema

Every multi-agent system has **one shared state object** that every node reads and updates. Getting this right is half the design.

For our **Customer Support Triage Agent**, the state should hold:

- the original `ticket` text
- what the Classifier decided (`category`, `urgency`)
- what the Retriever pulled (`retrieved` policies)
- what the Drafter wrote (`draft`)
- what QA said (`verdict`, plus a `revision_count` for our termination guard)
- a history of `revisions` — useful in observability and eval (Module 4 + 5)

In [1]:
from typing import TypedDict, Annotated, Literal
from operator import add
from pprint import pprint


In [2]:
class TriageState(TypedDict):
    # The raw ticket text from the customer.
    ticket: str

    # What the Classifier decides — bounded values, None until classified.
    category: Literal["billing", "technical", "account"] | None
    urgency:  Literal["low", "med", "high"] | None

    # What the Retriever pulled — list of dicts from the KB.
    retrieved: list[dict]

    # What the Drafter wrote.
    draft: str

    # What QA decided.
    verdict: Literal["pass", "revise"] | None

    # Termination guard counter.
    revision_count: int

    # Reducer pattern: each new revision is APPENDED, not replaced.
    # This is what makes 'revisions' accumulate across loop iterations.
    revisions: Annotated[list[str], add]


# Sanity check
initial_state: TriageState = {
    "ticket": "Why is my bill so high?",
    "category": None,
    "urgency": None,
    "retrieved": [],
    "draft": "",
    "verdict": None,
    "revision_count": 0,
    "revisions": [],
}
print(initial_state)


{'ticket': 'Why is my bill so high?', 'category': None, 'urgency': None, 'retrieved': [], 'draft': '', 'verdict': None, 'revision_count': 0, 'revisions': []}


**Hints**:

- `Literal["a", "b", "c"] | None` is great for category/urgency/verdict — bounded values, no typos.
- For the `revisions` list, use `Annotated[list[str], add]` so each node *appends* rather than *replaces*. This is the "reducer pattern" — Module 1's slides covered it.
- `revision_count` is a plain `int`; it starts at 0 and the QA node increments it.
- The `retrieved` list holds dicts from the KB, so type it as `list[dict]`.

## Part B  —  The graph topology

Module 1 slides 7–8 introduced four orchestration patterns. For the triage system, the design we converged on is:

**Sequential**, with **one feedback edge** from QA back to Drafter, and a hard `max_revisions = 2` guard.

Let's express it as Python data first — no LangGraph code yet. Just nodes, edges, and the routing logic.

In [3]:
MAX_REVISIONS = 2

# The four agent node names.
NODES = {"classify", "retrieve", "drafter", "qa"}

# Static edges — always fire in this order.
STATIC_EDGES = [
    ("classify", "retrieve"),
    ("retrieve", "drafter"),
    ("drafter", "qa"),
]

ENTRY_POINT = "classify"


def route_qa(state) -> str:
    """Conditional edge from the QA node.

    Returns the name of the next node — or 'END' if we should stop.
    """
    verdict = state["verdict"]
    count   = state["revision_count"]

    if verdict == "pass":
        return "END"

    # QA said 'revise' — but check the guard first.
    if count >= MAX_REVISIONS:
        # Module 1's 'missing termination' anti-pattern lives here if you forget this.
        return "END"

    return "drafter"


pprint({
    "nodes": NODES,
    "static_edges": STATIC_EDGES,
    "entry_point": ENTRY_POINT,
    "router_for_qa": route_qa.__name__,
})

{'entry_point': 'classify',
 'nodes': {'retrieve', 'qa', 'drafter', 'classify'},
 'router_for_qa': 'route_qa',
 'static_edges': [('classify', 'retrieve'),
                  ('retrieve', 'drafter'),
                  ('drafter', 'qa')]}


**Hints**:

- The `NODES` set is just the four agent names — strings.
- `STATIC_EDGES` are pairs that always fire in order: classify → retrieve → drafter → qa.
- `route_qa` is the conditional edge — it reads `state["verdict"]` and `state["revision_count"]` and returns the **next node name** (or `"END"` to stop).
- Don't forget the termination case: if `revision_count >= MAX_REVISIONS`, we go to `"END"` even if QA says revise. This is your guard against Module 1's "missing termination" anti-pattern.

## Part C  —  Anti-pattern check

Module 1 slide 10 named three design anti-patterns: **too many agents**, **missing termination**, **implicit state**.

For each scenario below, classify which anti-pattern is being committed (if any). Write your answer as a string in the dict below. Possible answers:
- `"too many agents"`
- `"missing termination"`
- `"implicit state"`
- `"no anti-pattern"`

In [4]:
scenarios = [
    {
        "id": "S1",
        "description": "Nine micro-agents, each doing ~5 lines of work.",
        # 9 boundaries where state can get lost, mostly tiny units of work.
        "anti_pattern": "too many agents",
    },
    {
        "id": "S2",
        "description": "Drafter references 'previous output' that's never in shared state.",
        # Classic implicit state — the data dependency exists but isn't typed.
        "anti_pattern": "implicit state",
    },
    {
        "id": "S3",
        "description": "Feedback loop with no max_iterations, costs blow up.",
        # No termination — the loop has no exit condition.
        "anti_pattern": "missing termination",
    },
    {
        "id": "S4",
        "description": "Three clear agents, typed state, sequential topology, explicit END.",
        # Looks healthy — no anti-pattern.
        "anti_pattern": "no anti-pattern",
    },
]

for s in scenarios:
    print(f"  {s['id']}:  {s['anti_pattern']}")

  S1:  too many agents
  S2:  implicit state
  S3:  missing termination
  S4:  no anti-pattern


## Part D  —  A small design decision

Two teams pitch the same triage system with slightly different topologies. Pick one and write a one-sentence justification.

**Topology A**:  classify → retrieve → drafter → qa  (feedback to drafter, max 2 revisions)

**Topology B**:  classify → retrieve → drafter → tone_checker → fact_checker → policy_checker (no feedback loop, 3 separate reviewer agents in sequence)

Which would you ship? Why?

In [5]:
# Topology A is the right answer for our scope.
CHOICE = "A"
REASON = (
    "B over-orchestrates — three separate reviewer agents add latency and "
    "boundary cost without a clear quality win, and there's no feedback "
    "loop so a bad draft can't be revised. A keeps a single QA reviewer "
    "with a bounded revision loop, which gives us a quality gate AND a "
    "hard termination."
)

print(f"I'd ship topology {CHOICE} because: {REASON}")

I'd ship topology A because: B over-orchestrates — three separate reviewer agents add latency and boundary cost without a clear quality win, and there's no feedback loop so a bad draft can't be revised. A keeps a single QA reviewer with a bounded revision loop, which gives us a quality gate AND a hard termination.


## Wrap up

That's Module 1. Three things to confirm before we move on:

1. **State** — every field is typed, no "maybe present" surprises.
2. **Topology** — every edge is either static or conditional with a clear router.
3. **Termination** — there's a hard ceiling on revisions; no infinite loops.

If those three are locked in, you're ready for **Module 2 — Building the End-to-End System with LangGraph**, where the design becomes runnable code.